# Задача 5. Экспериментальное исследование алгоритмов для регулярных запросов

### Постановка задачи (задача достижимости)
Дан граф $G$, $L$ - регулярный язык

Требуется найти $R' =  \{(v_i, v_j)| \pi(v_i \ v_j) \in L\}$

В данном эксперименте исследуются следующие задачи достижимости, решаемые в предыдущих работах:
- Достижимость между всеми парами вершин.

- Достижимость для каждой из заданного множества стартовых вершин.

В ходе данного эксперимента будут рассмотрены следующие вопросы, на которые будут получены ответы:
- Какое представление разреженных матриц и векторов лучше подходит для каждой из решаемых задач?

- Начиная с какого размера стартового множества выгоднее решать задачу для всех пар и выбирать нужные?

### Описание используемых алгоритмов
Для решения исследуемых задач были реализованы следующие алгоритмы:

- Достижимость между всеми парами вершин с использованием тензорного произведения $-$ строятся матрицы смежности графа и конечного автомата, затем тензорно перемножаются. Путем возведения в степень данной матрицы, находится транзитивное замыкание, по которому определяются искомые пары вершин. Реализация находится в `project/adjacency_matrix.py`

- Достижимость для каждой из заданного множества стартовых вершин с использованием идеи MS BFS. Реализация находится в `project/bfs_rpq.py`

В данных методах были использованы разряженные матрицы из библиотеки `scipy`

### Описание вычислительной машины и ОС

- Processor	AMD Ryzen 5 5600H
- RAM	16.0 GB (3000 MHz)
- OS Windows 11 Pro (64-bit), 23H2

### Описание окружения
- Python (CPython) 3.12.7
- All requirements are in `requierements.lock`

### Описание эксперимента

- Для ответа на первый вопрос были выбраны самые используемые матрицы из `scipy.sparse` $-$ `csr_matrix`, `dok_matrix`, `lil_matrix`, `csc_matrix`. Для каждого типа матриц оба алгоритма были запущены каждом графе и регулярном выражении. Из 20 запусков бралось среднее время работы и стандартное отклонение.

- Для ответа на второй вопрос, запускается также оба алогритма на каждом регулярном выражении, тип матрицы при этом выбран `csr_matrix`, как было в оригинальной реализации алгоритма, со случайно взятыми стартовыми вершинами в определенном процентном соотношении от общего числа вершин. Также из 20 запусков бралось среднее время работы с стандартное отклонение.

Все шаги и замеры представлены далее

### Импорты

In [ ]:
import logging
import time
from collections.abc import Collection
from typing import Optional

import cfpq_data
import matplotlib.pyplot as plt
import numpy as np
from networkx import MultiDiGraph
from pandas import DataFrame
from scipy.sparse import (
    csr_matrix,
    dok_matrix,
    lil_matrix,
    csc_matrix,
)

from project.adjacency_matrix import tensor_based_rpq
from project.bfs_rpq import ms_bfs_based_rpq
from project.graph_tools import GraphData

Отключить логи, выставить сид для генерации стартовых вершин

In [ ]:
seed = 42
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

### Описание набора данных для экспериментов

#### Графы
Для эксперемента были выбраны следующие графы:
 - pizza
 - skos
 - foaf
 - travel

 Graphs.svg

Такой выбор обусловлен большим разбросом в количестве меток, вершин и ребер в графах.
Ниже представлены все статические данные, необходимые для эксперимента

In [ ]:
graphs = [
    "pizza",
    "skos",
    "foaf",
    "travel",
]
matrix_types = [
    csr_matrix,
    csc_matrix,
    dok_matrix,
    lil_matrix,
]
launches = 20

raw_regex = [
    "(l1|l2)* l3",
    "(l3|l4)+ l1*",
    "l1 l2 l3 (l4|l1)*",
    "l1+ l2*",
]
rpqs = [
    tensor_based_rpq,
    ms_bfs_based_rpq,
]

persentages = [1, 2, 3, 5, 8, 12, 15, 20, 30, 40, 50, 60, 70, 80, 90]

Вспомогательные функции для проведения эксперимента
- `get_graph_by_name` - получить граф и его данные по имени
- `get_start_nodes` - получить стартовые надо с заданым отношением
- `make_grouped_bar_chart_plot` - построить и сохранить диаграмму по данным
- `plot_of_graph_data` - построить и сохранить диаграмму со статистикой по графам


In [ ]:
def get_graph_by_name(graph_name: str) -> tuple[MultiDiGraph, GraphData]:
    graph_path = cfpq_data.download(graph_name)
    graph_data = GraphData.get_graph_data_by_name(graph_name)
    return cfpq_data.graph_from_csv(graph_path), graph_data


def get_start_nodes(graph: MultiDiGraph, percent: float) -> set[int]:
    return cfpq_data.generate_multiple_source_percent(graph, percent, seed=seed)


def make_groped_bar_chart_plot(
    data: DataFrame,
    title: str,
    x_labels: Collection[str],
    y_label: str,
    error: Optional[DataFrame] = None,
):
    n_groups = len(data)
    n_bars = len(data.columns)
    indices = np.arange(n_groups)

    bar_width = 0.8 / n_bars
    fig, ax = plt.subplots(figsize=(10, 6))

    for i, col in enumerate(data.columns):
        offset = (i - (n_bars - 1) / 2) * bar_width
        values = data[col].values
        errors = None if error is None else error[col].values
        ax.bar(indices + offset, values, bar_width, yerr=errors, label=col, capsize=5)

    ax.set_xticks(indices)
    ax.set_xticklabels(x_labels, rotation=0, ha="right")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{title}.svg")


def plot_of_grahs_data(graphs: list[str]):
    graph_names = []
    node_counts = []
    edge_counts = []
    label_counts = []

    for graph_name in graphs:
        graph, data = get_graph_by_name(graph_name)
        graph_names.append(graph_name)
        node_counts.append(data.nodes_count)
        edge_counts.append(data.edges_count)
        label_counts.append(len(data.labels))

    graph_data = {
        "nodes": node_counts,
        "edges": edge_counts,
        "labels": label_counts,
    }
    make_groped_bar_chart_plot(DataFrame(graph_data), "Graphs", graph_names, "Count")

#### Запросы
В данном эксперементе были использованы следующие регулярные выражения в качестве запросов:

1. `(l1 | l2)* l3`
2. `(l3 | l4)+ l1*`
3. `l1 l2 l3 (l4|l1)*`
4. `l1+ l2*`

В данных запросах используются все общепринятые конструкции регулярных выражений (замыкание, конкатенация, альтернатива). В качестве меток $l_i$, для каждого графа были взяты наиболее часто встречающиеся, где $l_1$ - самая часто встречающаяся.

#### Код для построяния данных выражений

In [ ]:
def queries(labels: list[str]) -> list[str]:
    l1, l2, l3, l4, _ = labels
    return [
        f"({l1}|{l2})* {l3}",
        f"({l3}|{l4})+ {l1}*",
        f"{l1} {l2} {l3} ({l4}|{l1})*",
        f"({l1}+ {l2}*",
    ]

#### Этапы поиска ответа на первый вопрос
Замеры происходят следующим образом:

- Загружается граф из датасета
- Генерируются регулярные выражения
- Для каждого регулярного выражения и каждого типа матрицы, запускается выбранный алгоритм 20 раз
- Измеряется время выполнения каждого запуска
- Рассчитывается среднее время и стандартное отклонение
- Строятся графики для визуализации полученных данных, показывая время выполнения и отклонения для разных матричных представлений

Данные шаги повторяются для каждого графа и алгоритма

In [ ]:
def rpq_matrix_experiment():
    for graph_name in graphs:
        data = {}
        deviations = {}
        graph, graph_data = get_graph_by_name(graph_name)
        for matrix_type in matrix_types:
            for query in queries(graph_data.labels):
                for rpq in rpqs:
                    times = []
                    for _ in range(launches):
                        start = time.time()
                        rpq(query, graph, matrix_type=matrix_type)
                        times.append(time.time() - start)
                    average = np.mean(times)
                    deviation = np.std(times)
                    deviations.setdefault(
                        f"{matrix_type.__name__}({rpq.__name__})", []
                    ).append(deviation)
                    data.setdefault(
                        f"{matrix_type.__name__}({rpq.__name__})", []
                    ).append(average)

        make_groped_bar_chart_plot(
            DataFrame(data),
            f"RPQ {graph_name}",
            raw_regex,
            "Time",
            error=DataFrame(deviations),
        )

#### Функция для замеров и рисования графиков для вопроса №2
Замеры происходят следующим образом:

- Загружается граф из датасета
- Генерируются регулярные выражения
- Выбирается множество стартовых нод
- Для каждого регулярного выражения запускается выбранный алгоритм 20 раз
- Измеряется время выполнения каждого запуска
- Рассчитывается среднее время и стандартное отклонение
- Строятся диаграммы для визуализации полученных данных, показывая время выполнения и отклонения для разных матричных представлений

Данные шаги повторяются для каждого графа и алгоритма


In [ ]:
def rpq_start_nodes_experiment():
    for rpq in rpqs:
        for graph_name in graphs:
            data = {}
            deviations = {}
            graph, graph_data = get_graph_by_name(graph_name)
            matrix_type = csr_matrix
            for query in queries(graph_data.labels):
                for percent in persentages:
                    times = []
                    start_nodes = get_start_nodes(graph, percent)
                    for _ in range(launches):
                        start = time.time()
                        rpq(
                            query,
                            graph,
                            matrix_type=matrix_type,
                            start_nodes=start_nodes,
                            final_nodes=None,
                        )
                        times.append(time.time() - start)
                    average = np.mean(times)
                    deviation = np.std(times)
                    deviations.setdefault(f"{percent}", []).append(deviation)
                    data.setdefault(f"{percent}", []).append(average)

            make_groped_bar_chart_plot(
                DataFrame(data),
                f"RPQ {rpq.__name__} {graph_name}",
                raw_regex,
                "Time",
                error=DataFrame(deviations),
            )

## Вопрос №1. Какое представление разреженных матриц и векторов лучше подходит для каждой из решаемых задач?

### Сделаем замеры для `tensor_based_rpq`

In [ ]:
rpq_matrix_experiment()

RPQ skos.svgRPQ pizza.svgRPQ foaf.svgRPQ travel.svg

Как видно из диаграмм, лучшие результаты показали `csc_matrix` и `lil_matrix` для `ms_bfs`, однако для `tensor_based` выявить явную разницу в типе матрицы не удалось: на всех графах и запросах результаты довольно схожие, но есть девиации в меньшую строну у `csc_matrix` для графа `skos`.

# Вопрос №2. Начиная с какого размера стартового множества выгоднее решать задачу для всех пар и выбирать нужные?

## Данные для `tensor_based_rpq`
RPQ tensor_based_rpq pizza.svgRPQ tensor_based_rpq skos.svgRPQ tensor_based_rpq foaf.svgRPQ tensor_based_rpq travel.svg

Как видно из диаграмм для запросов $(l_1 | l_2)^* l_3$ и $l_1 l_2 l_3 (l_4 | l_1)^*$ рост времени линейный, однако для оставшихся двух запросов при переходе с 20% до 30% стартовых вершин прибавка времени резко увеличивается, так как шаг замера становится равным 10%, при этом темп роста сохраняется.

## Данные для `ms_bfs_based_rpq`
![RPQ ms_bfs_based_rpq pizza.svg](:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0idXRmLTgiIHN0YW5kYWxvbmU9Im5vIj8+CjwhRE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICAiaHR0cDovL3d3dy53My5vcmcvR3JhcGhpY3MvU1ZHLzEuMS9EVEQvc3ZnMTEuZHRkIj4KPHN2ZyB4bWxuczp4bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayIgd2lkdGg9IjcyMHB0IiBoZWlnaHQ9IjQzMnB0IiB2aWV3Qm94PSIwIDAgNzIwIDQzMiIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB2ZXJzaW9uPSIxLjEiPgogPG1ldGFkYXRhPgogIDxyZGY6UkRGIHhtbG5zOmRjPSJodHRwOi8vcHVybC5vcmcvZGMvZWxlbWVudHMvMS4xLyIgeG1sbnM6Y2M9Imh0dHA6Ly9jcmVhdGl2ZWNvbW1vbnMub3JnL25zIyIgeG1sbnM6cmRmPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5LzAyLzIyLXJkZi1zeW50YXgtbnMjIj4KICA…kZWZzPgogIDxjbGlwUGF0aCBpZD0icGNhNDgxMDY3ZWIiPgogICA8cmVjdCB4PSI0Ny43MiIgeT0iMjYuODgiIHdpZHRoPSI2NjEuNDgiIGhlaWdodD0iMzc2Ljg4Ii8+CiAgPC9jbGlwUGF0aD4KIDwvZGVmcz4KPC9zdmc+Cg==...)RPQ ms_bfs_based_rpq skos.svgRPQ ms_bfs_based_rpq foaf.svgRPQ ms_bfs_based_rpq travel.svg

Как видно из графиков, данные сильно отличаются от `tensor_based_rpq`, для каждого запроса рост времени остается линейным. Это может быть обусловлено тем, что алгоритм основан на MS BFS, где множество стартовых вершин используется для построения `front`, соответственно рост `front` будет также линейным.

# Выводы

### Вопрос №1. Какое представление разреженных матриц и векторов лучше подходит для каждой из решаемых задач?

Для `ms_bfs_based_rqp` удалось выявить, что наилучшим выбором будет `csr_matrix` и `dok_matrix`, так как они показали минимальное время для всех графов и запросов.

Для `tensor_based_rpq`` не удалось определить явных фаворитов, однако в результатах присутствуют несколько девиаций.

### Вопрос №2. Начиная с какого размера стартового множества выгоднее решать задачу для всех пар и выбирать нужные?

При использовании `ms_bfs_based_rqp` рост времени от кол-ва вершин всегда линеен, поэтому определить конкретное соотношение стартовых вершин довольно сложно.

`tensor_based_rpq` показал, что на некоторых запросах происходит явное увеличение темпа роста времени после 20% стартовых вершин, что может говорить о несовершенстве реализации алгоритма.